In [1]:
import sys
#sys.path.append('..') # append parent directory, we need it
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from torch.optim import lr_scheduler
import utils

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm
from utils.validation import get_validation_recalls
from models import helper

import models

In [2]:
import pytorch_lightning as pl
import torch
from torch.optim import lr_scheduler, optimizer
import wandb
from torch.utils.data import Subset
import utils
from models import helper
from models.helper import L2Norm, Flatten
from torchvision import transforms as T
from torch import nn
import torch

IMAGENET_MEAN_STD = {"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]}


class VPRModel(pl.LightningModule):
    """This is the main model for Visual Place Recognition
    we use Pytorch Lightning for modularity purposes.

    Args:
        pl (_type_): _description_
    """

    def __init__(
        self,
        # ---- Backbone
        backbone_arch="resnet50",
        backbone_config={},
        # ---- Aggregator
        agg_arch="ConvAP",
        agg_config={},
        # ---- Train hyperparameters
        lr=0.03,
        optimizer="sgd",
        weight_decay=1e-3,
        momentum=0.9,
        lr_sched="linear",
        lr_sched_args={
            "start_factor": 1,
            "end_factor": 0.2,
            "total_iters": 4000,
        },
        # ----- Loss
        loss_name="MultiSimilarityLoss",
        miner_name="MultiSimilarityMiner",
        miner_margin=0.1,
        faiss_gpu=True,
        args=None,
    ):
        super().__init__()

        # Backbone
        self.encoder_arch = backbone_arch
        self.backbone_config = backbone_config

        # Aggregator
        self.agg_arch = agg_arch
        self.agg_config = agg_config

        # Train hyperparameters
        self.lr = lr
        self.optimizer = optimizer
        self.weight_decay = weight_decay
        self.momentum = momentum
        self.lr_sched = lr_sched
        self.lr_sched_args = lr_sched_args

        # Loss
        self.loss_name = loss_name
        self.miner_name = miner_name
        self.miner_margin = miner_margin

        self.save_hyperparameters()  # write hyperparams into a file

        self.loss_fn = utils.get_loss(loss_name)
        self.miner = utils.get_miner(miner_name, miner_margin)
        self.batch_acc = (
            []
        )  # we will keep track of the % of trivial pairs/triplets at the loss level

        self.faiss_gpu = faiss_gpu
        self.args = args
        # ----------------------------------
        # get the backbone and the aggregator
        self.backbone = helper.get_backbone(backbone_arch, backbone_config)

        if "netvlad" in agg_arch.lower():
            # Create an instance of the AggConfig class
            # agg_config = AggConfig(agg_config)
            self.aggLayer = helper.get_aggregator(agg_arch, agg_config)

            # cluster using gsv single city
            if agg_config.initialize_clusters:
                from dataloaders.GSVCitiesDataset import GSVCitiesDataset
                # Instantiate GSVCitiesDataset with the desired city
                selected_city = "London"  # Replace with the city you want
                single_city_dataset = GSVCitiesDataset(
                    cities=[selected_city],
                    img_per_place=1,  # Adjust as needed
                    min_img_per_place=1,  # Adjust as needed
                    random_sample_from_each_place=True,
                    transform=T.Compose(
                        [
                            T.Resize(
                                self.args.resize,
                                interpolation=T.InterpolationMode.BILINEAR,
                            ),
                            T.ToTensor(),
                            T.Normalize(
                                mean=IMAGENET_MEAN_STD["mean"],
                                std=IMAGENET_MEAN_STD["std"],
                            ),  # Adjust mean and std if needed
                        ]
                    ),
                )
                self.aggLayer.initialize_netvlad_layer(
                    agg_config, single_city_dataset, self.backbone
                )

            if agg_config.l2 == "before_pool":
                self.aggLayer = nn.Sequential(L2Norm(), self.aggLayer, Flatten())
            elif agg_config.l2 == "after_pool":
                self.aggLayer = nn.Sequential(self.aggLayer, L2Norm(), Flatten())
            elif agg_config.l2 == "onlyFlatten":
                self.aggLayer = nn.Sequential(self.aggLayer, Flatten())

            if (
                agg_config.useFC
            ):  # fc_dim used so have a NV agg layer in nn.Seq aggregation layer
                if agg_config.nv_pca is not None:
                    netvlad_output_dim = agg_config.nv_pca

                netvlad_output_dim *= agg_config.clusters_num

                if agg_config.fc_output_dim == 0:
                    fcLayer = nn.Identity()
                    agg_config.fc_output_dim = netvlad_output_dim
                else:
                    fcLayer = nn.Linear(netvlad_output_dim, agg_config.fc_output_dim)
                    agg_config.fc_output_dim = agg_config.fc_output_dim

                self.aggregator = nn.Sequential(self.aggLayer, fcLayer, L2Norm())

            else:  # no fc_dim used
                self.aggregator = self.aggLayer
                del self.aggLayer

                # check wpca layer to be used? / can be used during evaluation only
                if agg_config.wpca:
                    if args.nv_pca is not None:
                        netvlad_output_dim = args.nv_pca
                    else:
                        netvlad_output_dim = agg_config.dim
                    netvlad_output_dim = agg_config.clusters_num * netvlad_output_dim
                    pca_conv = nn.Conv2d(
                        netvlad_output_dim,
                        agg_config.num_pcs,
                        kernel_size=(1, 1),
                        stride=1,
                        padding=0,
                    )
                    self.WPCA = nn.Sequential(*[pca_conv, Flatten(), L2Norm(dim=-1)])

        else:  # SALAD
            self.aggregator = helper.get_aggregator(agg_arch, agg_config)

            # check wpca layer to be used? / can be used during evaluation only
            if args.wpca:
                salad_output_dim = args.num_clusters * args.cluster_dim + args.token_dim
                pca_conv = nn.Conv2d(
                    salad_output_dim,
                    args.num_pcs,
                    kernel_size=(1, 1),
                    stride=1,
                    padding=0,
                )
                self.WPCA = nn.Sequential(*[pca_conv, Flatten(), L2Norm(dim=-1)])
        # For validation in Lightning v2.0.0
        self.val_outputs = []

    # the forward pass of the lightning model
    def forward(self, x):
        x = self.backbone(x)
        x = self.aggregator(x)
        return x

    # configure the optimizer
    def configure_optimizers(self):
        if self.optimizer.lower() == "sgd":
            optimizer = torch.optim.SGD(
                self.parameters(),
                lr=self.lr,
                weight_decay=self.weight_decay,
                momentum=self.momentum,
            )
        elif self.optimizer.lower() == "adamw":
            optimizer = torch.optim.AdamW(
                self.parameters(), lr=self.lr, weight_decay=self.weight_decay
            )
        elif self.optimizer.lower() == "adam":
            optimizer = torch.optim.AdamW(
                self.parameters(), lr=self.lr, weight_decay=self.weight_decay
            )
        else:
            raise ValueError(
                f'Optimizer {self.optimizer} has not been added to "configure_optimizers()"'
            )

        if self.lr_sched.lower() == "multistep":
            scheduler = lr_scheduler.MultiStepLR(
                optimizer,
                milestones=self.lr_sched_args["milestones"],
                gamma=self.lr_sched_args["gamma"],
            )
        elif self.lr_sched.lower() == "cosine":
            scheduler = lr_scheduler.CosineAnnealingLR(
                optimizer, self.lr_sched_args["T_max"]
            )
        elif self.lr_sched.lower() == "linear":
            scheduler = lr_scheduler.LinearLR(
                optimizer,
                start_factor=self.lr_sched_args["start_factor"],
                end_factor=self.lr_sched_args["end_factor"],
                total_iters=self.lr_sched_args["total_iters"],
            )

        return [optimizer], [scheduler]

    # configure the optizer step, takes into account the warmup stage
    def optimizer_step(self, epoch, batch_idx, optimizer, optimizer_closure):
        # warm up lr
        optimizer.step(closure=optimizer_closure)
        self.lr_schedulers().step()

    #  The loss function call (this method will be called at each training iteration)
    def loss_function(self, descriptors, labels):
        # we mine the pairs/triplets if there is an online mining strategy
        if self.miner is not None:
            miner_outputs = self.miner(descriptors, labels)
            loss = self.loss_fn(descriptors, labels, miner_outputs)

            # calculate the % of trivial pairs/triplets
            # which do not contribute in the loss value
            nb_samples = descriptors.shape[0]
            nb_mined = len(set(miner_outputs[0].detach().cpu().numpy()))
            batch_acc = 1.0 - (nb_mined / nb_samples)

        else:  # no online mining
            loss = self.loss_fn(descriptors, labels)
            batch_acc = 0.0
            if type(loss) == tuple:
                # somes losses do the online mining inside (they don't need a miner objet),
                # so they return the loss and the batch accuracy
                # for example, if you are developping a new loss function, you might be better
                # doing the online mining strategy inside the forward function of the loss class,
                # and return a tuple containing the loss value and the batch_accuracy (the % of valid pairs or triplets)
                loss, batch_acc = loss

        # keep accuracy of every batch and later reset it at epoch start
        self.batch_acc.append(batch_acc)
        # log it
        self.log(
            "b_acc",
            sum(self.batch_acc) / len(self.batch_acc),
            prog_bar=True,
            logger=True,
        )
        if not self.args.no_wandb:
            wandb.log({"b_acc": sum(self.batch_acc) / len(self.batch_acc)})

        return loss

    # This is the training step that's executed at each iteration
    def training_step(self, batch, batch_idx):
        places, labels = batch

        # Note that GSVCities yields places (each containing N images)
        # which means the dataloader will return a batch containing BS places
        BS, N, ch, h, w = places.shape

        # reshape places and labels
        images = places.view(BS * N, ch, h, w)
        labels = labels.view(-1)

        # Feed forward the batch to the model
        descriptors = self(
            images
        )  # Here we are calling the method forward that we defined above

        if torch.isnan(descriptors).any():
            raise ValueError("NaNs in descriptors")

        loss = self.loss_function(
            descriptors, labels
        )  # Call the loss_function we defined above

        self.log("loss", loss.item(), logger=True, prog_bar=True)
        if not self.args.no_wandb:
            wandb.log({"loss": loss.item()})
        return {"loss": loss}

    def on_train_epoch_end(self):
        # we empty the batch_acc list for next epoch
        self.batch_acc = []

    # For validation, we will also iterate step by step over the validation set
    # this is the way Pytorch Lghtning is made. All about modularity, folks.
    def validation_step(self, batch, batch_idx, dataloader_idx=None):
        places, _ = batch
        descriptors = self(places)
        self.val_outputs[dataloader_idx].append(descriptors.detach().cpu())
        return descriptors.detach().cpu()

    def on_validation_epoch_start(self):
        # reset the outputs list
        self.val_outputs = [
            [] for _ in range(len(self.trainer.datamodule.val_datasets))
        ]

    def on_validation_epoch_end(self):
        """this return descriptors in their order
        depending on how the validation dataset is implemented
        for this project (MSLS val, Pittburg val), it is always references then queries
        [R1, R2, ..., Rn, Q1, Q2, ...]
        """
        val_step_outputs = self.val_outputs

        dm = self.trainer.datamodule
        # The following line is a hack: if we have only one validation set, then
        # we need to put the outputs in a list (Pytorch Lightning does not do it presently)
        if len(dm.val_datasets) == 1:  # we need to put the outputs in a list
            val_step_outputs = [val_step_outputs]

        for i, (val_set_name, val_dataset) in enumerate(
            zip(dm.val_set_names, dm.val_datasets)
        ):
            feats = torch.concat(val_step_outputs[i], dim=0)

            if "pitts" in val_set_name:
                # split to ref and queries
                num_references = val_dataset.dbStruct.numDb
                positives = val_dataset.getPositives()
            elif "msls" in val_set_name:
                # split to ref and queries
                num_references = val_dataset.num_references
                positives = val_dataset.pIdx
            else:
                print(f"Please implement validation_epoch_end for {val_set_name}")
                raise NotImplemented

            r_list = feats[:num_references]
            q_list = feats[num_references:]
            distances, predictions, pitts_dict = utils.get_validation_recalls(
                r_list=r_list,
                q_list=q_list,
                k_values=[1, 5, 10],  # , 15, 20, 50, 100],
                gt=positives,
                print_results=True,
                dataset_name=val_set_name,
                faiss_gpu=self.faiss_gpu,
            )
            del r_list, q_list, feats, num_references, positives

            self.log(f"{val_set_name}/R1", pitts_dict[1], prog_bar=False, logger=True)
            self.log(f"{val_set_name}/R5", pitts_dict[5], prog_bar=False, logger=True)
            self.log(f"{val_set_name}/R10", pitts_dict[10], prog_bar=False, logger=True)
            if not self.args.no_wandb:
                metrics = {
                    f"{val_set_name}-val/Recall@{k}": v for k, v in pitts_dict.items()
                }
                wandb.log(metrics)

        print("\n\n")

        # reset the outputs list
        self.val_outputs = []

In [4]:
import torch
import pytorch_lightning as pl
from torch.utils.data import DataLoader
import torchvision.transforms as T
from tqdm import tqdm
import argparse
import wandb
import random
import numpy as np
from collections import OrderedDict
from utils.validation import get_validation_recalls
import os

# Dataloader
from dataloaders.val.NordlandDataset import NordlandDataset
from dataloaders.val.EssexDataset import EssexDataset
# from dataloaders.val.MapillaryDataset import MSLS
# from dataloaders.val.MapillaryTestDataset import MSLSTest
# from dataloaders.val.PittsburghDataset import PittsburghDataset
# from dataloaders.val.SPEDDataset import SPEDDataset
# from dataloaders.val.StluciaDataset import StluciaDataset
# from dataloaders.val.Tokyo247Dataset import Tokyo247Dataset
# from dataloaders.val.AmstertimeDataset import AmstertimeDataset
# from dataloaders.val.BaiduDataset import BaiduDataset
# from dataloaders.val.SfsmDataset import SfsmDataset

VAL_DATASETS = [
    "MSLS",
    "MSLS_Test",
    "pitts30k_test",
    "pitts250k_test",
    "Nordland",
    "ESSEX",
    "SPED",
    "pitts30k_val",
    "st_lucia",
    "tokyo247",
    "amstertime",
    "baidu",
    "sfsm",
]


def input_transform(image_size=None):
    MEAN = [0.485, 0.456, 0.406]
    STD = [0.229, 0.224, 0.225]
    if image_size:
        print(f"image_size === ===== {image_size}")
        return T.Compose(
            [
                T.Resize(image_size, interpolation=T.InterpolationMode.BILINEAR),
                T.ToTensor(),
                T.Normalize(mean=MEAN, std=STD),
            ]
        )
    else:
        return T.Compose([T.ToTensor(), T.Normalize(mean=MEAN, std=STD)])


def get_val_dataset(dataset_name, image_size=None):
    dataset_name = dataset_name.lower()
    transform = input_transform(image_size=image_size)

    if "nordland" in dataset_name:
        ds = NordlandDataset(input_transform=transform)

    elif "msls_test" in dataset_name:
        ds = MSLSTest(input_transform=transform)
    elif 'essex' in dataset_name:
        ds = EssexDataset(input_transform = transform)
    elif "msls" in dataset_name:
        ds = MSLS(input_transform=transform)

    elif "pitts" in dataset_name:
        ds = PittsburghDataset(which_ds=dataset_name, input_transform=transform)

    elif "sped" in dataset_name:
        ds = SPEDDataset(input_transform=transform)

    elif "st_lucia" in dataset_name:
        ds = StluciaDataset(input_transform=transform)

    elif "tokyo247" in dataset_name:
        ds = Tokyo247Dataset(input_transform=transform)

    elif "sfsm" in dataset_name:
        ds = SfsmDataset(input_transform=transform)

    elif "amstertime" in dataset_name:
        ds = AmstertimeDataset(input_transform=transform)

    elif "baidu" in dataset_name:
        ds = BaiduDataset(input_transform=transform)

    else:
        raise ValueError

    num_references = ds.num_references
    num_queries = ds.num_queries
    ground_truth = ds.ground_truth
    return ds, num_references, num_queries, ground_truth


def get_pca_encoding(model, vlad_encoding):
    pca_encoding = model.WPCA(vlad_encoding.unsqueeze(-1).unsqueeze(-1))
    return pca_encoding


def get_descriptors(model, dataloader):
    descriptors = []
    with torch.no_grad():
        with torch.autocast(device_type=args.device, dtype=torch.float16):
            for batch in tqdm(dataloader, "Calculating descritptors..."):
                imgs, labels = batch
                imgs = imgs.to(args.device)

                if args.useFC:  # using fc_output_dim
                    output = model(imgs).cpu()
                else:  # no fc layer after NV layer / vanilla NV
                    if not args.storeSAB and not args.storeSOTL:
                        vlad_encoding = model(imgs)
                    else:
                        image_encoding = model.backbone(imgs)
                        store_path = os.path.join(
                            os.path.dirname(os.path.dirname(args.resume_train)),
                            val_name,
                        )
                        vlad_encoding = model.aggregator(
                            image_encoding,
                            labels=labels.cpu().numpy(),
                            dirPath=store_path,
                        )
                        del image_encoding
                    if args.wpca:
                        vlad_encoding = get_pca_encoding(model, vlad_encoding)
                    output = vlad_encoding.cpu()  # .detach().cpu().numpy()
                descriptors.append(output)
                del imgs

    return torch.cat(descriptors)


def replace_key(k, num):
    for old, new in {"WPCA_" + str(num) + ".": "WPCA."}.items():
        if old in k:
            k = k.replace(old, new)
    return k


def load_model():
    if "netvlad" in args.aggregation.lower():
        agg_config = args
        useToken = False
    elif "salad" in args.aggregation.lower():
        agg_config = {
            "num_channels": args.num_channels,
            "num_clusters": args.num_clusters,
            "cluster_dim": args.cluster_dim,
            "token_dim": args.token_dim,
            "expName": args.expName,
            "reduce_feature_dims": args.reduce_feature_dims,
            "reduce_token_dims": args.reduce_token_dims,
            "args": args,
        }
        useToken = True
    model = VPRModel(
        backbone_arch=args.backbone,
        backbone_config={
            "num_trainable_blocks": args.num_trainable_blocks,
            "return_token": useToken,
            "norm_layer": args.norm_layer,
        },
        agg_arch=args.aggregation,
        agg_config=agg_config,
        args=args,
    )

    if args.ckpt_state_dict:
        checkpoint = torch.load(args.resume_train)
        if args.wpca:
            checkpoint["state_dict"] = OrderedDict(
                {
                    replace_key(k, args.num_pcs): v
                    for k, v in checkpoint["state_dict"].items()
                }
            )
            model_state_dict = model.state_dict()
            # Filter out keys that are not in the model's current state dict
            checkpoint["state_dict"] = {
                k: v
                for k, v in checkpoint["state_dict"].items()
                if k in model_state_dict
            }
        model.load_state_dict(checkpoint["state_dict"])
    else:
        model.load_state_dict(torch.load(args.resume_train))
    model = model.eval()
    model = model.to(args.device)
    print(f"Loaded model from {args.resume_train} Successfully!")
    return model

    
args_dict = {
    "dataset_name": 'gsv_cities', 
    "save_dir": './', 
    "expName": 'dnv2_NV_AB', 
    "batch_size": 2, 
    "img_per_place": 4, 
    "min_img_per_place": 4, 
    "shuffle_all": False, 
    "random_sample_from_each_place": True, 
    "resize": (322, 322), 
    "num_workers": 20, 
    "show_data_stats": True, 
    "backbone": 'vbdino', 
    "num_trainable_blocks": 4, 
    "norm_layer": True, 
    "aggregation": 'NETVLAD', 
    "in_dim": 2048, 
    "out_dim": 512, 
    "p": 3, 
    "in_channels": 2048, 
    "out_channels": 512, 
    "in_h": 20, 
    "in_w": 20, 
    "mix_depth": 1, 
    "storeSOTL": False, 
    "num_channels": 768, 
    "num_clusters": 64, 
    "cluster_dim": 128, 
    "token_dim": 256, 
    "reduce_feature_dims": True, 
    "reduce_token_dims": True, 
    "l2": 'none', 
    "forLoopAlt": True, 
    "fc_output_dim": 512, 
    "dim": 768, 
    "clusters_num": 64, 
    "initialize_clusters": False, 
    "useFC": False, 
    "nv_pca": None, 
    "nv_pca_randinit": False, 
    "nv_pca_alt": False, 
    "nv_pca_alt_mlp": False, 
    "infer_batch_size": 16, 
    "storeSAB": False, 
    "antiburst": True, 
    "ab_w": 8.0, 
    "ab_b": 7.0, 
    "ab_p": 1.0, 
    "ab_gen": None, 
    "ab_relu": False, 
    "ab_soft": False, 
    "ab_inv": False, 
    "ab_t": None, 
    "ab_testOnly": False, 
    "ab_allFreezeButAb": False, 
    "ab_fixed": False, 
    "ab_kp": None, 
    "ab_wOnly": False, 
    "device": 'cuda', 
    "resume_train": './saved_models/vlad_buff/dnv2_NV_AB_wpca8192_last.ckpt', 
    "ckpt_state_dict": True, 
    "val_datasets": ['ESSEX', 'Nordland'], 
    "pl_seed": True, 
    "wpca": True, 
    "seed": 1, 
    "num_pcs": 8192, 
    "no_wandb": True, 
    "store_eval_output": False
}

args=argparse.Namespace(**args_dict)

# Check: dont have wpca for systems having fc layer after NV layer
if args.useFC:
    assert args.wpca == False

if not args.no_wandb:
    dataset_name = args.dataset_name.lower()[:4]
    wandb_dataStr = dataset_name
    args.expName = "eval-" + wandb_dataStr + args.expName

if args.pl_seed:
    pl.seed_everything(seed=int(args.seed), workers=True)

model = load_model()
print(f"<=======backbone_arch: {args.backbone}========>")
print(f"<=======aggregation: {args.aggregation}========>")
print(f"<=======args: {args}========>")
# print(f"<=======model: {model}========>")

if not args.no_wandb:
    wandb.init(project="vlad_buff", config=args)
    # update runName
    runName = wandb.run.name
    wandb.run.name = args.expName + "-" + runName.split("-")[-1]
    wandb.run.save()

for val_name in args.val_datasets:
    print(f"<=======expName: {args.expName}========>")

    val_dataset, num_references, num_queries, ground_truth = get_val_dataset(
        val_name, args.resize
    )
    val_loader = DataLoader(
        val_dataset,
        num_workers=args.num_workers,
        batch_size=args.batch_size,
        shuffle=False,
        pin_memory=True,
    )
    print(f"Evaluating on {val_name}")
    descriptors = get_descriptors(model, val_loader)

    print(f"Descriptor dimension {descriptors.shape[1]}")
    r_list = descriptors[:num_references]
    q_list = descriptors[num_references:]

    print("total_size", descriptors.shape[0], num_queries + num_references)
    print(f"Queries:{num_queries}, References:{num_references}")

    recalls_dict, preds = get_validation_recalls(
        r_list=r_list,
        q_list=q_list,
        k_values=[1, 5, 10],  # , 15, 20, 25],
        gt=ground_truth,
        print_results=True,
        dataset_name=val_name,
        faiss_gpu=False,
    )
    del descriptors

Seed set to 1
Using cache found in /home/kirilltobola/.var/app/com.visualstudio.code/cache/torch/hub/facebookresearch_dinov2_main
/home/kirilltobola/.var/app/com.visualstudio.code/cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/kirilltobola/.var/app/com.visualstudio.code/cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/kirilltobola/.var/app/com.visualstudio.code/cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Loaded model from ./saved_models/vlad_buff/dnv2_NV_AB_wpca8192_last.ckpt Successfully!
<=======backbone_arch: vbdino========>
<=======aggregation: NETVLAD========>
<=======args: Namespace(dataset_name='gsv_cities', save_dir='./', expName='dnv2_NV_AB', batch_size=2, img_per_place=4, min_img_per_place=4, shuffle_all=False, random_sample_from_each_place=True, resize=(322, 322), num_workers=20, show_data_stats=True, backbone='vbdino', num_trainable_blocks=4, norm_layer=True, aggregation='NETVLAD', in_dim=2048, out_dim=512, p=3, in_channels=2048, out_channels=512, in_h=20, in_w=20, mix_depth=1, storeSOTL=False, num_channels=768, num_clusters=64, cluster_dim=128, token_dim=256, reduce_feature_dims=True, reduce_token_dims=True, l2='none', forLoopAlt=True, fc_output_dim=512, dim=768, clusters_num=64, initialize_clusters=False, useFC=False, nv_pca=None, nv_pca_randinit=False, nv_pca_alt=False, nv_pca_alt_mlp=False, infer_batch_size=16, storeSAB=False, antiburst=True, ab_w=8.0, ab_b=7.0, ab_p=1.

/home/kirilltobola/.conda/envs/vpr-env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 20 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Evaluating on ESSEX


Calculating descritptors...: 100%|██████████| 210/210 [00:11<00:00, 18.34it/s]


Descriptor dimension 8192
total_size 420 420
Queries:210, References:210


+------------------------------------+
|        Performance on ESSEX        |
+----------+-------+--------+--------+
|    K     |   1   |   5    |   10   |
+----------+-------+--------+--------+
| Recall@K | 91.90 | 100.00 | 100.00 |
+----------+-------+--------+--------+
<=======expName: dnv2_NV_AB========>
image_size === ===== (322, 322)
Evaluating on Nordland


Calculating descritptors...: 100%|██████████| 15176/15176 [11:50<00:00, 21.35it/s]


Descriptor dimension 8192
total_size 30352 30352
Queries:2760, References:27592


+----------------------------------+
|     Performance on Nordland      |
+----------+-------+-------+-------+
|    K     |   1   |   5   |   10  |
+----------+-------+-------+-------+
| Recall@K | 74.17 | 87.64 | 91.05 |
+----------+-------+-------+-------+
